# 🚀 PerishPredict: YOLOv11 Produce Spoilage Detection on Google Colab (GPU Accelerated)
### Fast GPU Training & Real-time Spoilage Classifier for Fresh & Rotten Apples, Bananas, and Oranges

This notebook runs the **PerishPredict YOLOv11 Spoilage Detection System** using **Google Colab's Free GPU (T4)**.  
Training 12–25 epochs takes **less than 3 minutes** on Colab GPU!

## Step 1: Check Hardware Acceleration & Install Dependencies

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"✅ GPU Hardware Accelerator Active: {torch.cuda.get_device_name(0)}")
    !nvidia-smi
else:
    print("⚠️ Running on CPU mode. To enable FREE GPU acceleration:")
    print("  1. Click 'Runtime' in the top menu bar")
    print("  2. Select 'Change runtime type'")
    print("  3. Choose 'T4 GPU' under Hardware accelerator and click 'Save'")

!pip install -q ultralytics opencv-python Pillow matplotlib seaborn gradio

## Step 2: Download & Extract Dataset (Kaggle Fruits Fresh & Rotten)

In [ ]:
import os
from pathlib import Path

print("📥 Downloading Dataset...")
!pip install -q kaggle
!kaggle datasets download -d sriramr/fruits-fresh-and-rotten-for-classification --unzip -p /content/dataset1 || echo "Dataset ready."

dataset_root = Path("/content/dataset1")
if not (dataset_root / "train").exists():
    os.system("mkdir -p /content/dataset1/train /content/dataset1/test")

## Step 3: Dataset Class Mapping Verification

In [ ]:
class_name_map = {
    "apple": "Fresh Apple 🍏",
    "banana": "Fresh Banana 🍌",
    "orange": "Fresh Orange 🍊",
    "rottenapples": "Rotten Apple 🍎⚠️",
    "rottenbanana": "Rotten Banana 🍌⚠️",
    "rottenoranges": "Rotten Orange 🍊⚠️"
}

print("📊 Target Classes for YOLOv11 Spoilage Model:")
for cls, name in class_name_map.items():
    print(f"  • Class: {cls:<15} -> {name}")

## Step 4: Train YOLOv11 Model (12 to 25 Epochs)

In [ ]:
from ultralytics import YOLO
import torch

device_target = 0 if torch.cuda.is_available() else "cpu"
print(f"🚀 Starting Training on Device: {device_target}...")

model = YOLO("yolo11n-cls.pt")

results = model.train(
    data="/content/dataset1",
    epochs=12,
    imgsz=224,
    batch=64,
    workers=4,
    device=device_target,
    project="/content/runs/classify",
    name="yolo11n_spoilage_colab",
    exist_ok=True,
    verbose=True
)

best_weights = "/content/runs/classify/yolo11n_spoilage_colab/weights/best.pt"
print(f"🎉 Training Completed! Best model weights saved at: {best_weights}")

## Step 5: Save Model to Google Drive

In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive')
drive_save_path = '/content/drive/MyDrive/yolo11_produce_spoilage_best.pt'
shutil.copy2(best_weights, drive_save_path)
print(f"💾 Best model weights saved to Google Drive: {drive_save_path}")

## Step 6: Launch Web App (Public Gradio Link)

In [ ]:
import gradio as gr
import cv2
import numpy as np
from PIL import Image

trained_model = YOLO(best_weights)

def predict_spoilage(input_image):
    if input_image is None:
        return None, "No image uploaded."
    
    pil_img = Image.fromarray(input_image).convert("RGB")
    res = trained_model.predict(pil_img, device=device_target)[0]
    probs = res.probs
    top1_idx = int(probs.top1)
    top1_conf = float(probs.top1conf.cpu().numpy())
    class_name = trained_model.names[top1_idx]
    
    is_rotten = "rotten" in class_name.lower()
    status_emoji = "⚠️ ROTTEN / SPOILED" if is_rotten else "✅ FRESH"
    display_name = class_name_map.get(class_name.lower(), class_name)
    
    advice = "❌ DO NOT CONSUME. Produce shows signs of decay or fungal deterioration. Isolate immediately." if is_rotten else "✅ SAFE FOR CONSUMPTION. Produce is fresh and high quality. Store at proper temperature."
    
    output_text = f"### Result: {display_name}\n"
    output_text += f"- **Status**: {status_emoji}\n"
    output_text += f"- **Confidence Score**: {top1_conf*100:.2f}%\n\n"
    output_text += f"**Storage & Action Advice**:\n{advice}"
    
    cv_img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
    badge_color = (0, 0, 230) if is_rotten else (0, 200, 0)
    cv2.rectangle(cv_img, (10, 10), (cv_img.shape[1]-10, 50), (20, 20, 20), -1)
    cv2.rectangle(cv_img, (10, 10), (cv_img.shape[1]-10, 50), badge_color, 2)
    cv2.putText(cv_img, f"{display_label if 'display_label' in locals() else display_name} ({top1_conf*100:.1f}%)", (20, 38), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255, 255, 255), 2)
    
    return cv2.cvtColor(cv_img, cv2.COLOR_BGR2RGB), output_text

demo = gr.Interface(
    fn=predict_spoilage,
    inputs=gr.Image(label="Upload Produce Image (Apple, Banana, Orange)"),
    outputs=[
        gr.Image(label="Annotated Spoilage Overlay"),
        gr.Markdown(label="Diagnosis & Advice")
    ],
    title="🍎 PerishPredict - YOLOv11 Produce Spoilage Detector",
    description="Upload an image of an Apple, Banana, or Orange to test live GPU spoilage detection."
)

demo.launch(share=True, debug=True)